In [1]:
import sys, subprocess, textwrap, platform

print("📌 Active Notebook Python:")
print("Executable:", sys.executable)
print("Version   :", sys.version.split()[0])
print("Platform  :", platform.platform())
print("-" * 70)

def run(cmd):
    print("\n▶", " ".join(cmd))
    return subprocess.run(cmd, check=False, capture_output=True, text=True)

def pip_install(pkgs):
    # pkgs can be a string or list
    if isinstance(pkgs, str):
        pkgs = [pkgs]
    cmd = [sys.executable, "-m", "pip", "install", "-U"] + pkgs
    r = run(cmd)
    if r.returncode != 0:
        print("❌ pip failed.\nSTDOUT:\n", r.stdout[-1500:], "\nSTDERR:\n", r.stderr[-1500:])
        return False
    print("✅ pip success.\n", r.stdout[-1000:])
    return True

# 1) Upgrade pip tooling (safe)
ok = pip_install(["pip", "setuptools", "wheel"])

# 2) Install core scientific stack WITHOUT pinning incompatible numpy
# (This avoids the Python 3.13 + numpy==1.23.5 incompatibility)
ok = ok and pip_install(["numpy", "scipy", "pandas", "scikit-learn", "joblib"])

# 3) Try to install model deps
lgb_ok = pip_install("lightgbm")
shap_ok = pip_install("shap")

print("\n" + "=" * 70)
if lgb_ok and shap_ok:
    print("✅ Environment ready in this kernel (Python 3.13).")
    print("🔁 Restart kernel now, then run:")
    print("   import lightgbm as lgb, shap; print(lgb.__version__, shap.__version__)")
else:
    print("⚠️ Could not install LightGBM and/or SHAP into Python 3.13.")
    print("This is normal on Windows when wheels aren't available yet for 3.13.")
    print("\n✅ BEST FIX: switch the notebook kernel to Python 3.9 (where you already have them installed).")
    print("\nRun these in Command Prompt / PowerShell (once):\n")
    print(textwrap.dedent(rf"""
    {r"C:\Users\HP\AppData\Local\Programs\Python\Python39\python.exe"} -m pip install --upgrade ipykernel
    {r"C:\Users\HP\AppData\Local\Programs\Python\Python39\python.exe"} -m ipykernel install --user --name python39 --display-name "Python 3.9 (BiteBot)"
    """).strip())
    print("\nThen in Jupyter: Kernel → Change Kernel → Python 3.9 (BiteBot)")
    print("=" * 70)


📌 Active Notebook Python:
Executable: c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe
Version   : 3.13.2
Platform  : Windows-11-10.0.26200-SP0
----------------------------------------------------------------------

▶ c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe -m pip install -U pip setuptools wheel
✅ pip success.


▶ c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe -m pip install -U numpy scipy pandas scikit-learn joblib
✅ pip success.
 quirement already satisfied: joblib in c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages (1.5.3)
Using cached numpy-2.4.2-cp313-cp313-win_amd64.whl (12.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5


▶ c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe -m pip install -U lightgbm
✅ pip success.


▶ c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe -m pip in

In [2]:
# FINAL FIX CELL: Repair SciPy/SHAP import mismatch (Python 3.13)
import sys, subprocess

def run(cmd):
    print("▶", " ".join(cmd))
    subprocess.check_call(cmd)

py = sys.executable
print("Notebook Python:", py)

# 1) Remove potentially broken SciPy + SHAP stack
run([py, "-m", "pip", "uninstall", "-y", "scipy", "shap", "numba", "llvmlite"])

# 2) Reinstall in a known-good order (forces consistent wheels)
#    - numpy first
#    - then scipy
#    - then numba/llvmlite
#    - then shap
run([py, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
run([py, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy"])
run([py, "-m", "pip", "install", "--upgrade", "--force-reinstall", "scipy"])
run([py, "-m", "pip", "install", "--upgrade", "--force-reinstall", "llvmlite", "numba"])
run([py, "-m", "pip", "install", "--upgrade", "--force-reinstall", "shap"])

print("\n✅ Done. Now: Kernel → Restart Kernel, then run:")
print('   import scipy, shap; print("scipy", scipy.__version__, "shap", shap.__version__)')


Notebook Python: c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe
▶ c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe -m pip uninstall -y scipy shap numba llvmlite
▶ c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip setuptools wheel
▶ c:\Users\HP\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade --force-reinstall numpy


KeyboardInterrupt: 

In [ ]:

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Any, Tuple, List
from pathlib import Path
import joblib

import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:

@dataclass
class ClinicalModelConfig:
    random_state: int = 42
    test_size: float = 0.2

    feature_cols: Tuple[str, ...] = (
        "age",
        "sex_male",
        "has_htn",
        "has_dm",
        "has_ckd",
        "serum_sodium",
        "serum_potassium",
        "creatinine",
        "egfr",
        "hba1c",
        "fbs",
        "sbp",
        "dbp",
        "bmi",
    )

    targets: Tuple[str, ...] = (
        "sodium_sensitivity",
        "potassium_sensitivity",
        "protein_restriction",
    )

    lgbm_params: Dict[str, Any] = None

    model_dir: str = "artifacts/models"
    encoder_dir: str = "artifacts/encoders"

    def __post_init__(self):
        if self.lgbm_params is None:
            self.lgbm_params = dict(
                objective="multiclass",
                num_class=3,
                learning_rate=0.05,
                n_estimators=600,
                num_leaves=31,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                random_state=self.random_state,
                verbose=-1,
            )


In [ ]:
# ===============================
# Monotonic Constraints
# ===============================

def build_monotonic_constraints(features: List[str]) -> List[int]:
    """
    +1 : higher value => higher dietary restriction risk
    -1 : higher value => lower risk
     0 : no constraint
    """
    constraints = []
    for f in features:
        f = f.lower()
        if f in {"creatinine", "serum_potassium", "hba1c", "fbs", "sbp", "dbp", "bmi"}:
            constraints.append(+1)
        elif f in {"egfr"}:
            constraints.append(-1)
        elif f in {"serum_sodium"}:
            constraints.append(+1)
        else:
            constraints.append(0)
    return constraints


In [ ]:
# ===============================
# Clinical Risk Stratifier
# ===============================

class ClinicalRiskStratifier:

    def __init__(self, cfg: ClinicalModelConfig):
        self.cfg = cfg
        self.models: Dict[str, lgb.LGBMClassifier] = {}
        self.imputer = None
        self.scaler = None

    def fit(self, df: pd.DataFrame):
        X = df[self.cfg.feature_cols]
        Y = df[self.cfg.targets]

        self.imputer = SimpleImputer(strategy="median")
        X = self.imputer.fit_transform(X)

        self.scaler = StandardScaler()
        X = self.scaler.fit_transform(X)

        mono = build_monotonic_constraints(list(self.cfg.feature_cols))

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            Y,
            test_size=self.cfg.test_size,
            random_state=self.cfg.random_state,
            stratify=Y[self.cfg.targets[0]],
        )

        for target in self.cfg.targets:
            model = lgb.LGBMClassifier(**self.cfg.lgbm_params)
            model.set_params(monotone_constraints=mono)
            model.fit(X_train, y_train[target])

            self.models[target] = model

            preds = model.predict(X_test)
            print(f"\n=== Evaluation: {target} ===")
            print(classification_report(y_test[target], preds))
            print(confusion_matrix(y_test[target], preds))

    def predict(self, clinical_input: Dict[str, Any]) -> Dict[str, str]:
        X = pd.DataFrame([clinical_input], columns=self.cfg.feature_cols)
        X = self.imputer.transform(X)
        X = self.scaler.transform(X)

        label_map = {0: "low", 1: "moderate", 2: "high"}

        return {
            target: label_map[int(model.predict(X)[0])]
            for target, model in self.models.items()
        }


In [ ]:
class LabelGenerator:
    """Generate nutrition risk labels from clinical features"""

    def __init__(self, config: ClinicalModelConfig):
        self.config = config

    def generate_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Generate multi-label targets based on clinical thresholds
        Replace with actual labeled data in production
        """
        print("\n" + "="*80)
        print("GENERATING CLINICAL RISK LABELS")
        print("="*80)

        df_labeled = df.copy()

        # 1. Sodium sensitivity
        print("\n1. Sodium Sensitivity:")
        df_labeled['sodium_sensitivity'] = 0  # Default: low

        sodium_col = 'sodium_mean' if 'sodium_mean' in df.columns else None
        if sodium_col:
            # Moderate: high sodium OR hypertension
            mask_moderate = (df_labeled[sodium_col] > 145) | \
                           (df_labeled.get('hypertension', 0) == 1)
            df_labeled.loc[mask_moderate, 'sodium_sensitivity'] = 1

            # High: very high sodium OR (hypertension + BP meds)
            mask_high = (df_labeled[sodium_col] > 150) | \
                       ((df_labeled.get('hypertension', 0) == 1) &
                        (df_labeled.get('ace_inhibitors', 0) == 1))
            df_labeled.loc[mask_high, 'sodium_sensitivity'] = 2

            counts = df_labeled['sodium_sensitivity'].value_counts().sort_index()
            print(f"   Low: {counts.get(0, 0)}, Moderate: {counts.get(1, 0)}, High: {counts.get(2, 0)}")
        else:
            print("   ⚠ Sodium data not available, using defaults")

        # 2. Potassium sensitivity
        print("\n2. Potassium Sensitivity:")
        df_labeled['potassium_sensitivity'] = 0

        k_col = 'potassium_mean' if 'potassium_mean' in df.columns else None
        egfr_col = 'egfr' if 'egfr' in df.columns else None

        if k_col and egfr_col:
            # Moderate: high K OR reduced eGFR
            mask_moderate = (df_labeled[k_col] > 4.5) | (df_labeled[egfr_col] < 60)
            df_labeled.loc[mask_moderate, 'potassium_sensitivity'] = 1

            # High: very high K OR (CKD + severely reduced eGFR)
            mask_high = (df_labeled[k_col] > 5.0) | \
                       ((df_labeled.get('chronic_kidney_disease', 0) == 1) &
                        (df_labeled[egfr_col] < 30))
            df_labeled.loc[mask_high, 'potassium_sensitivity'] = 2

            counts = df_labeled['potassium_sensitivity'].value_counts().sort_index()
            print(f"   Low: {counts.get(0, 0)}, Moderate: {counts.get(1, 0)}, High: {counts.get(2, 0)}")
        else:
            print("   ⚠ Potassium/eGFR data not available, using defaults")

        # 3. Protein restriction
        print("\n3. Protein Restriction:")
        df_labeled['protein_restriction'] = 0

        if egfr_col:
            bun_col = 'bun_mean' if 'bun_mean' in df.columns else None
            creat_col = 'creatinine_mean' if 'creatinine_mean' in df.columns else None

            # Moderate: reduced eGFR OR elevated BUN
            mask_moderate = (df_labeled[egfr_col] < 60)
            if bun_col:
                mask_moderate |= (df_labeled[bun_col] > 20)
            df_labeled.loc[mask_moderate, 'protein_restriction'] = 1

            # High: severely reduced eGFR OR CKD OR very high creatinine
            mask_high = (df_labeled[egfr_col] < 30) | \
                       (df_labeled.get('chronic_kidney_disease', 0) == 1)
            if creat_col:
                mask_high |= (df_labeled[creat_col] > 2.0)
            df_labeled.loc[mask_high, 'protein_restriction'] = 2

            counts = df_labeled['protein_restriction'].value_counts().sort_index()
            print(f"   Low: {counts.get(0, 0)}, Moderate: {counts.get(1, 0)}, High: {counts.get(2, 0)}")
        else:
            print("   ⚠ eGFR data not available, using defaults")

        return df_labeled

In [ ]:
class ClinicalRiskModel:
    """Multi-label clinical risk stratification model"""

    def __init__(self, config: ClinicalModelConfig):
        self.config = config
        self.models = {}
        self.feature_names = None
        self.monotone_constraints = None

    def define_monotonic_constraints(self, feature_names: List[str]) -> List[int]:
        """Define monotonic constraints based on clinical knowledge"""
        constraints = {feature: 0 for feature in feature_names}

        # Clinical monotonic relationships
        if 'egfr' in constraints:
            constraints['egfr'] = -1  # Lower eGFR → Higher risk
        if 'creatinine_mean' in constraints:
            constraints['creatinine_mean'] = 1
        if 'bun_mean' in constraints:
            constraints['bun_mean'] = 1
        if 'age' in constraints:
            constraints['age'] = 1

        # Disease indicators
        for disease in ['hypertension', 'diabetes_mellitus',
                       'chronic_kidney_disease', 'heart_failure']:
            if disease in constraints:
                constraints[disease] = 1

        return [constraints[feat] for feat in feature_names]

    def train_single_target(self, X_train, y_train, X_val, y_val,
                           target_name, monotone_constraints):
        """Train model for single target"""
        print(f"\nTraining: {target_name}")
        print(f"  Class distribution: {np.bincount(y_train)}")

        # Class weights for imbalance
        class_counts = np.bincount(y_train)
        class_weights = len(y_train) / (len(class_counts) * class_counts)
        sample_weights = class_weights[y_train]

        model = lgb.LGBMClassifier(
            **self.config.LGBM_PARAMS,
            monotone_constraints=monotone_constraints
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric='multi_logloss',
            sample_weight=sample_weights,
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        return model

    def train(self, X_train, y_train, X_val, y_val, feature_names):
        """Train all models"""
        print("\n" + "="*80)
        print("TRAINING MODELS")
        print("="*80)

        self.feature_names = feature_names
        self.monotone_constraints = self.define_monotonic_constraints(feature_names)

        for target in self.config.TARGETS:
            self.models[target] = self.train_single_target(
                X_train, y_train[target].values,
                X_val, y_val[target].values,
                target, self.monotone_constraints
            )

        print("\n✓ All models trained")

    def predict(self, X):
        """Get predictions"""
        return {target: model.predict(X)
                for target, model in self.models.items()}

    def predict_proba(self, X):
        """Get probability predictions"""
        return {target: model.predict_proba(X)
                for target, model in self.models.items()}

In [ ]:
def main():
    """Execute complete training pipeline"""

    config = ClinicalModelConfig()
    config.MODEL_DIR.mkdir(exist_ok=True)
    config.RESULTS_DIR.mkdir(exist_ok=True)

    print("="*80)
    print("CLINICAL RISK STRATIFICATION - MIMIC-IV-Ext-CEKG")
    print("="*80)

    # ========================================
    # 1. LOAD DATA
    # ========================================
    loader = MIMICDataLoader(config)

    # Sample 1000 patients for testing (remove in production)
    success = loader.load_data(sample_patients=1000)

    if not success or loader.eventlog is None:
        print("\n✗ Failed to load event log. Exiting.")
        return

    # ========================================
    # 2. FEATURE EXTRACTION
    # ========================================
    extractor = ClinicalFeatureExtractor(config)

    print("\n" + "="*80)
    print("FEATURE EXTRACTION")
    print("="*80)

    # Extract features from different sources
    lab_features = extractor.extract_lab_features(
        loader.eventlog, loader.activity_attrs
    )

    vital_features = extractor.extract_vital_signs(
        loader.eventlog, loader.activity_attrs
    )

    demographics = extractor.extract_demographics(loader.entities_attrs)

    disease_indicators = extractor.extract_disease_indicators(
        loader.eventlog, loader.icd_data
    )

    medication_flags = extractor.extract_medication_flags(loader.eventlog)

    # ========================================
    # 3. MERGE FEATURES
    # ========================================
    print("\n" + "="*80)
    print("MERGING FEATURES")
    print("="*80)

    # Start with patient-admission pairs
    feature_dfs = [df for df in [lab_features, vital_features, disease_indicators,
                                  medication_flags] if len(df) > 0]

    if len(feature_dfs) == 0:
        print("✗ No features extracted. Exiting.")
        return

    # Merge on Entity1_ID and Entity2_ID
    merged_features = feature_dfs[0]
    for df in feature_dfs[1:]:
        merged_features = merged_features.merge(
            df, on=['Entity1_ID', 'Entity2_ID'], how='outer'
        )

    # Add demographics (patient-level)
    if len(demographics) > 0:
        merged_features = merged_features.merge(
            demographics, on='Entity1_ID', how='left'
        )

    print(f"\n✓ Merged features shape: {merged_features.shape}")
    print(f"✓ Columns: {merged_features.columns.tolist()[:10]}... ({len(merged_features.columns)} total)")

    # ========================================
    # 4. COMPUTE DERIVED FEATURES
    # ========================================
    merged_features = extractor.compute_derived_features(merged_features)

    # ========================================
    # 5. GENERATE LABELS
    # ========================================
    label_generator = LabelGenerator(config)
    labeled_data = label_generator.generate_labels(merged_features)

    # ========================================
    # 6. PREPROCESSING
    # ========================================
    print("\n" + "="*80)
    print("PREPROCESSING")
    print("="*80)

    # Separate features and labels
    feature_cols = [col for col in labeled_data.columns
                   if col not in config.TARGETS + ['Entity1_ID', 'Entity2_ID']]

    X = labeled_data[feature_cols].copy()
    y = labeled_data[config.TARGETS].copy()

    # Handle missing values
    print(f"\nMissing values before imputation: {X.isnull().sum().sum()}")

    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(
        imputer.fit_transform(X),
        columns=X.columns,
        index=X.index
    )

    print(f"Missing values after imputation: {X_imputed.isnull().sum().sum()}")

    # Remove rows with missing labels
    valid_mask = ~y.isnull().any(axis=1)
    X_clean = X_imputed[valid_mask]
    y_clean = y[valid_mask]

    print(f"\n✓ Final dataset: {X_clean.shape[0]} samples, {X_clean.shape[1]} features")
    print(f"✓ Label distribution:")
    for target in config.TARGETS:
        counts = y_clean[target].value_counts().sort_index()
        print(f"   {target}: {counts.to_dict()}")


    # ========================================
    # 7. TRAIN/VAL/TEST SPLIT
    # ========================================
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean,
        test_size=config.TEST_SIZE,
        random_state=config.RANDOM_STATE,
        stratify=y_clean[config.TARGETS[0]]
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train,
        test_size=0.15,
        random_state=config.RANDOM_STATE,
        stratify=y_train[config.TARGETS[0]]
    )

    print(f"\nData split:")
    print(f"  Train: {len(X_train)}")
    print(f"  Val: {len(X_val)}")
    print(f"  Test: {len(X_test)}")

    # ========================================
    # 8. MODEL TRAINING
    # ========================================
    model = ClinicalRiskModel(config)
    model.train(
        X_train.values, y_train,
        X_val.values, y_val,
        X_train.columns.tolist()
    )

    # ========================================
    # 9. EVALUATION
    # ========================================
    print("\n" + "="*80)
    print("EVALUATION")
    print("="*80)

    predictions = model.predict(X_test.values)
    predictions_proba = model.predict_proba(X_test.values)

    for target in config.TARGETS:
        print(f"\n{'='*60}")
        print(f"Target: {target.upper()}")
        print(f"{'='*60}")

        y_true = y_test[target].values
        y_pred = predictions[target]
        y_proba = predictions_proba[target]

        # Classification report
        print(classification_report(
            y_true, y_pred,
            target_names=['Low', 'Moderate', 'High'],
            zero_division=0
        ))

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Low', 'Moderate', 'High'],
                   yticklabels=['Low', 'Moderate', 'High'])
        plt.title(f'Confusion Matrix: {target}')
        plt.ylabel('True')
        plt.xlabel('Predicted')
        plt.tight_layout()
        plt.savefig(config.RESULTS_DIR / f'confusion_matrix_{target}.png')
        plt.close()

        # Feature importance
        importance = model.models[target].feature_importances_
        top_idx = np.argsort(importance)[-15:]

        plt.figure(figsize=(10, 6))
        plt.barh(range(len(top_idx)), importance[top_idx])
        plt.yticks(range(len(top_idx)),
                  [X_train.columns[i] for i in top_idx])
        plt.xlabel('Importance')
        plt.title(f'Top 15 Features: {target}')
        plt.tight_layout()
        plt.savefig(config.RESULTS_DIR / f'feature_importance_{target}.png')
        plt.close()

    # ========================================
    # 10. SAVE MODELS
    # ========================================
    print("\n" + "="*80)
    print("SAVING MODELS")
    print("="*80)

    for target, lgb_model in model.models.items():
        model_path = config.MODEL_DIR / f"model_{target}.pkl"
        joblib.dump(lgb_model, model_path)
        print(f"✓ Saved: {model_path}")

    # Save preprocessor and feature names
    joblib.dump(imputer, config.MODEL_DIR / "imputer.pkl")
    joblib.dump(X_train.columns.tolist(), config.MODEL_DIR / "feature_names.pkl")

    print("\n" + "="*80)
    print("✓ PIPELINE COMPLETE")
    print("="*80)
    print(f"\nModels: {config.MODEL_DIR}")
    print(f"Results: {config.RESULTS_DIR}")


In [ ]:
# ===============================
# Save / Load
# ===============================

def save(self):
    Path(self.cfg.model_dir).mkdir(parents=True, exist_ok=True)
    Path(self.cfg.encoder_dir).mkdir(parents=True, exist_ok=True)

    for t, m in self.models.items():
        joblib.dump(m, f"{self.cfg.model_dir}/{t}.joblib")

    joblib.dump(self.imputer, f"{self.cfg.encoder_dir}/imputer.joblib")
    joblib.dump(self.scaler, f"{self.cfg.encoder_dir}/scaler.joblib")

def load(self):
    self.models = {
        t: joblib.load(f"{self.cfg.model_dir}/{t}.joblib")
        for t in self.cfg.targets
    }
    self.imputer = joblib.load(f"{self.cfg.encoder_dir}/imputer.joblib")
    self.scaler = joblib.load(f"{self.cfg.encoder_dir}/scaler.joblib")


In [ ]:
# ===============================
# Example Inference
# ===============================

cfg = ClinicalModelConfig()
model = ClinicalRiskStratifier(cfg)

# df_train should come from MIMIC-IV + guideline-based labels
# model.fit(df_train)

patient = {
    "age": 55,
    "sex_male": 1,
    "has_htn": 1,
    "has_dm": 1,
    "has_ckd": 1,
    "serum_sodium": 144,
    "serum_potassium": 5.6,
    "creatinine": 2.3,
    "egfr": 28,
    "hba1c": 8.4,
    "fbs": 170,
    "sbp": 152,
    "dbp": 94,
    "bmi": 29,
}

# risk_states = model.predict(patient)
# print(risk_states)
